In [1]:
import pandas as pd

df = pd.read_csv('../data/results.csv')

print("Shape:", df.shape)
print("\nDate range:", df['date'].min(), "to", df['date'].max())
print("\nMissing values:\n", df.isnull().sum())
print("\nTournament types:", df['tournament'].nunique())

Shape: (49477, 9)

Date range: 1872-11-30 to 2026-06-27

Missing values:
 date           0
home_team      0
away_team      0
home_score    52
away_score    52
tournament     0
city           0
country        0
neutral        0
dtype: int64

Tournament types: 200


In [2]:
# Drop rows with no score (future matches)
df = df.dropna(subset=['home_score', 'away_score'])

# Convert date to datetime
df['date'] = pd.to_datetime(df['date'])

print("Clean shape:", df.shape)
print("Rows dropped:", 49477 - df.shape[0])

Clean shape: (49425, 9)
Rows dropped: 52


In [3]:
# See top 20 most common tournaments
print(df['tournament'].value_counts().head(20))

tournament
Friendly                                18388
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                            984
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
CFU Caribbean Cup qualification           606
Merdeka Tournament                        599
British Home Championship                 523
CONCACAF Nations League                   422
AFC Asian Cup                             421
Gold Cup                                  420
Gulf Cup                                  410
Island Games                              394
UEFA Euro                                 388
Asian Games                               368
Name: count, dtype: int64


In [4]:
# Keep only matches from 1990 onwards
df_modern = df[df['date'].dt.year >= 1990]

print("Modern era shape:", df_modern.shape)
print("Date range:", df_modern['date'].min(), "to", df_modern['date'].max())

Modern era shape: (32307, 9)
Date range: 1990-01-12 00:00:00 to 2026-06-16 00:00:00


In [5]:
# Assign importance weights to tournament types
def get_tournament_weight(tournament):
    if 'World Cup' in tournament and 'qualification' not in tournament:
        return 3.0  # Highest — actual WC matches
    elif 'qualification' in tournament:
        return 2.0  # High — competitive qualifiers
    elif tournament in ['UEFA Euro', 'Copa América', 'Africa Cup of Nations',
                        'AFC Asian Cup', 'UEFA Nations League', 'CONCACAF Nations League']:
        return 2.0  # High — major tournaments
    elif tournament == 'Friendly':
        return 0.5  # Low — friendlies don't mean much
    else:
        return 1.0  # Medium — other competitive

df_modern['tournament_weight'] = df_modern['tournament'].apply(get_tournament_weight)
print(df_modern['tournament_weight'].value_counts())

tournament_weight
2.0    14588
0.5    10829
1.0     6258
3.0      632
Name: count, dtype: int64


In [6]:
df_modern = df_modern.copy()
df_modern['year'] = df_modern['date'].dt.year
df_modern['total_goals'] = df_modern['home_score'] + df_modern['away_score']
df_modern['goal_diff'] = df_modern['home_score'] - df_modern['away_score']

print(df_modern[['home_score', 'away_score', 'total_goals', 'goal_diff']].describe().round(2))

       home_score  away_score  total_goals  goal_diff
count    32307.00    32307.00     32307.00   32307.00
mean         1.65        1.11         2.76       0.55
std          1.71        1.35         1.99       2.36
min          0.00        0.00         0.00     -21.00
25%          0.00        0.00         1.00      -1.00
50%          1.00        1.00         2.00       0.00
75%          2.00        2.00         4.00       2.00
max         31.00       21.00        31.00      31.00


In [7]:
print("Most common scorelines:")
scorelines = df_modern.groupby(['home_score', 'away_score']).size().sort_values(ascending=False).head(10)
print(scorelines)

Most common scorelines:
home_score  away_score
1.0         0.0           3582
            1.0           3338
0.0         0.0           2849
2.0         0.0           2660
            1.0           2485
0.0         1.0           2411
1.0         2.0           1664
3.0         0.0           1587
0.0         2.0           1568
2.0         2.0           1222
dtype: int64


In [8]:
def get_team_form(team, date, df, n=10):
    """
    Get a team's average goals scored and conceded
    in their last n matches before a given date.
    """
    # Get all matches involving this team before the given date
    home_matches = df[(df['home_team'] == team) & (df['date'] < date)].copy()
    away_matches = df[(df['away_team'] == team) & (df['date'] < date)].copy()
    
    # Rename columns so we can combine them
    home_matches = home_matches.rename(columns={
        'home_score': 'scored', 'away_score': 'conceded'
    })
    away_matches = away_matches.rename(columns={
        'away_score': 'scored', 'home_score': 'conceded'
    })
    
    # Combine home and away matches
    all_matches = pd.concat([
        home_matches[['date', 'scored', 'conceded', 'tournament_weight']],
        away_matches[['date', 'scored', 'conceded', 'tournament_weight']]
    ]).sort_values('date').tail(n)
    
    if len(all_matches) == 0:
        return 1.0, 1.0  # Default if no history
    
    avg_scored = all_matches['scored'].mean()
    avg_conceded = all_matches['conceded'].mean()
    
    return round(avg_scored, 3), round(avg_conceded, 3)

In [9]:
# Test: Brazil's form before the 2022 World Cup Final
scored, conceded = get_team_form('Brazil', pd.Timestamp('2022-12-01'), df_modern)
print(f"Brazil avg scored: {scored}, avg conceded: {conceded}")

# Test: England
scored, conceded = get_team_form('England', pd.Timestamp('2022-12-01'), df_modern)
print(f"England avg scored: {scored}, avg conceded: {conceded}")

Brazil avg scored: 3.0, avg conceded: 0.3
England avg scored: 1.6, avg conceded: 1.2


In [10]:
from tqdm.notebook import tqdm
tqdm.pandas()

rows = []

for _, match in tqdm(df_modern.iterrows(), total=len(df_modern)):
    home = match['home_team']
    away = match['away_team']
    date = match['date']

    h_scored, h_conceded = get_team_form(home, date, df_modern)
    a_scored, a_conceded = get_team_form(away, date, df_modern)

    rows.append({
        'date':               date,
        'home_team':          home,
        'away_team':          away,
        'home_avg_scored':    h_scored,
        'home_avg_conceded':  h_conceded,
        'away_avg_scored':    a_scored,
        'away_avg_conceded':  a_conceded,
        'is_neutral':         int(match['neutral']),
        'tournament_weight':  match['tournament_weight'],
        'home_score':         match['home_score'],
        'away_score':         match['away_score'],
    })

df_features = pd.DataFrame(rows)
print(df_features.shape)
print(df_features.head(3))

  0%|          | 0/32307 [00:00<?, ?it/s]

(32307, 11)
        date home_team away_team  home_avg_scored  home_avg_conceded  \
0 1990-01-12   Algeria      Mali              1.0                1.0   
1 1990-01-14   Algeria  Cameroon              5.0                0.0   
2 1990-01-17    Greece   Belgium              1.0                1.0   

   away_avg_scored  away_avg_conceded  is_neutral  tournament_weight  \
0              1.0                1.0           1                0.5   
1              1.0                1.0           1                0.5   
2              1.0                1.0           0                0.5   

   home_score  away_score  
0         5.0         0.0  
1         3.0         1.0  
2         2.0         0.0  


In [11]:
df_features.to_csv('../data/features.csv', index=False)
print("Saved!")

Saved!


In [12]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

df_features = pd.read_csv('../data/features.csv')

FEATURES = [
    'home_avg_scored', 'home_avg_conceded',
    'away_avg_scored', 'away_avg_conceded',
    'is_neutral', 'tournament_weight'
]

X = df_features[FEATURES]
y_home = df_features['home_score']
y_away = df_features['away_score']

# 80% train, 20% test — NO shuffling (time-series data!)
X_train, X_test, yh_train, yh_test, ya_train, ya_test = train_test_split(
    X, y_home, y_away, test_size=0.2, shuffle=False
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (25845, 6)
Test size: (6462, 6)


In [13]:
# Train
model_home = LinearRegression()
model_away = LinearRegression()

model_home.fit(X_train, yh_train)
model_away.fit(X_train, ya_train)

# Predict
pred_home = model_home.predict(X_test)
pred_away = model_away.predict(X_test)

# Evaluate
mae_home = mean_absolute_error(yh_test, pred_home)
mae_away = mean_absolute_error(ya_test, pred_away)

print(f"Home Goals MAE: {mae_home:.3f}")
print(f"Away Goals MAE: {mae_away:.3f}")

Home Goals MAE: 1.128
Away Goals MAE: 0.886


In [14]:
# Predict: Brazil vs Germany on neutral ground
test_match = pd.DataFrame([{
    'home_avg_scored': 3.0,
    'home_avg_conceded': 0.3,
    'away_avg_scored': 2.1,
    'away_avg_conceded': 0.8,
    'is_neutral': 1,
    'tournament_weight': 3.0
}])

home_pred = model_home.predict(test_match)[0]
away_pred = model_away.predict(test_match)[0]

print(f"Brazil vs Germany → {home_pred:.1f} - {away_pred:.1f}")

Brazil vs Germany → 2.0 - 1.0


In [15]:
# Dummy baseline: always predict the mean
dummy_home_pred = np.full_like(yh_test, yh_train.mean(), dtype=float)
dummy_away_pred = np.full_like(ya_test, ya_train.mean(), dtype=float)

dummy_mae_home = mean_absolute_error(yh_test, dummy_home_pred)
dummy_mae_away = mean_absolute_error(ya_test, dummy_away_pred)

print(f"Dummy (mean) MAE  → Home: {dummy_mae_home:.3f}, Away: {dummy_mae_away:.3f}")
print(f"Our model MAE     → Home: {mae_home:.3f}, Away: {mae_away:.3f}")

Dummy (mean) MAE  → Home: 1.237, Away: 0.933
Our model MAE     → Home: 1.128, Away: 0.886


In [16]:
from sklearn.ensemble import RandomForestRegressor

rf_home = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)
rf_away = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)

rf_home.fit(X_train, yh_train)
rf_away.fit(X_train, ya_train)

rf_pred_home = rf_home.predict(X_test)
rf_pred_away = rf_away.predict(X_test)

rf_mae_home = mean_absolute_error(yh_test, rf_pred_home)
rf_mae_away = mean_absolute_error(ya_test, rf_pred_away)

print(f"Random Forest MAE → Home: {rf_mae_home:.3f}, Away: {rf_mae_away:.3f}")
print(f"Linear Reg MAE    → Home: {mae_home:.3f}, Away: {mae_away:.3f}")
print(f"Dummy MAE         → Home: {dummy_mae_home:.3f}, Away: {dummy_mae_away:.3f}")

Random Forest MAE → Home: 1.135, Away: 0.878
Linear Reg MAE    → Home: 1.128, Away: 0.886
Dummy MAE         → Home: 1.237, Away: 0.933


In [17]:
import pandas as pd

importance = pd.DataFrame({
    'feature': FEATURES,
    'importance': rf_home.feature_importances_
}).sort_values('importance', ascending=False)

print(importance)

             feature  importance
3  away_avg_conceded    0.642459
0    home_avg_scored    0.150018
1  home_avg_conceded    0.131395
2    away_avg_scored    0.041886
5  tournament_weight    0.030659
4         is_neutral    0.003583


In [18]:
def calculate_elo_ratings(df, k=20, initial_rating=1500):
    """
    Calculate Elo ratings for every team after every match.
    Returns a dict mapping team -> current rating,
    and adds elo_home/elo_away columns to df (rating BEFORE the match).
    """
    elo = {}
    home_elo_list = []
    away_elo_list = []
    
    for _, row in df.iterrows():
        home = row['home_team']
        away = row['away_team']
        
        # Get current ratings (default 1500 if team unseen)
        r_home = elo.get(home, initial_rating)
        r_away = elo.get(away, initial_rating)
        
        # Save the PRE-match rating as a feature (no data leakage)
        home_elo_list.append(r_home)
        away_elo_list.append(r_away)
        
        # Expected outcome
        e_home = 1 / (1 + 10 ** ((r_away - r_home) / 400))
        e_away = 1 - e_home
        
        # Actual outcome
        if row['home_score'] > row['away_score']:
            actual_home, actual_away = 1.0, 0.0
        elif row['home_score'] < row['away_score']:
            actual_home, actual_away = 0.0, 1.0
        else:
            actual_home, actual_away = 0.5, 0.5
        
        # Update ratings
        elo[home] = r_home + k * (actual_home - e_home)
        elo[away] = r_away + k * (actual_away - e_away)
    
    df['home_elo'] = home_elo_list
    df['away_elo'] = away_elo_list
    return df, elo

In [19]:
df_modern_sorted = df_modern.sort_values('date').reset_index(drop=True)
df_modern_sorted, final_elo = calculate_elo_ratings(df_modern_sorted)

print(df_modern_sorted[['date', 'home_team', 'away_team', 'home_elo', 'away_elo']].tail(10))

            date     home_team    away_team     home_elo     away_elo
32297 2026-06-14   Netherlands        Japan  1848.438194  1853.561731
32298 2026-06-14        Sweden      Tunisia  1686.300665  1669.368010
32299 2026-06-15       Belgium        Egypt  1825.935901  1723.762626
32300 2026-06-15          Iran  New Zealand  1802.132178  1599.281929
32301 2026-06-15         Spain   Cape Verde  1957.336494  1589.629936
32302 2026-06-15  Saudi Arabia      Uruguay  1639.628971  1800.727457
32303 2026-06-16        France      Senegal  1915.313835  1779.707869
32304 2026-06-16          Iraq       Norway  1682.705702  1740.395399
32305 2026-06-16     Argentina      Algeria  1970.031685  1785.549271
32306 2026-06-16       Austria       Jordan  1754.152458  1648.825419


In [20]:
teams_to_check = ['Brazil', 'Argentina', 'France', 'England', 'Germany', 'Spain']
for team in teams_to_check:
    print(f"{team}: {final_elo.get(team, 'Not found'):.1f}")

Brazil: 1898.9
Argentina: 1975.2
France: 1921.6
England: 1876.5
Germany: 1866.5
Spain: 1949.5


In [21]:
FEATURES_V2 = [
    'home_avg_scored', 'home_avg_conceded',
    'away_avg_scored', 'away_avg_conceded',
    'is_neutral', 'tournament_weight',
    'home_elo', 'away_elo'
]
df_features['date'] = pd.to_datetime(df_features['date'])
# Merge Elo columns into df_features using date+teams as the join key
df_features_v2 = df_features.merge(
    df_modern_sorted[['date', 'home_team', 'away_team', 'home_elo', 'away_elo']],
    on=['date', 'home_team', 'away_team'],
    how='left'
)

print(df_features_v2.shape)
print(df_features_v2[FEATURES_V2].isnull().sum())

(32309, 13)
home_avg_scored      0
home_avg_conceded    0
away_avg_scored      0
away_avg_conceded    0
is_neutral           0
tournament_weight    0
home_elo             0
away_elo             0
dtype: int64


In [22]:
# Find rows where merge key isn't unique in df_modern_sorted
dupe_keys = df_modern_sorted[df_modern_sorted.duplicated(
    subset=['date', 'home_team', 'away_team'], keep=False
)]
print(dupe_keys[['date', 'home_team', 'away_team', 'home_score', 'away_score']])

            date  home_team       away_team  home_score  away_score
32228 2026-06-06  Gibraltar  Cayman Islands         4.0         1.0
32233 2026-06-06  Gibraltar  Cayman Islands         4.0         1.0


In [23]:
print(df_features.duplicated(subset=['date', 'home_team', 'away_team']).sum())

1


In [24]:
df_modern_sorted = df_modern_sorted.drop_duplicates(
    subset=['date', 'home_team', 'away_team', 'home_score', 'away_score']
).reset_index(drop=True)

print("Shape after dedup:", df_modern_sorted.shape)

Shape after dedup: (32306, 15)


In [25]:
df_modern_sorted, final_elo = calculate_elo_ratings(df_modern_sorted)

print(df_modern_sorted[['date', 'home_team', 'away_team', 'home_elo', 'away_elo']].tail(5))

            date     home_team away_team     home_elo     away_elo
32301 2026-06-15  Saudi Arabia   Uruguay  1639.628971  1800.727457
32302 2026-06-16        France   Senegal  1915.313835  1779.707869
32303 2026-06-16          Iraq    Norway  1682.705702  1740.395399
32304 2026-06-16     Argentina   Algeria  1970.031685  1785.549271
32305 2026-06-16       Austria    Jordan  1754.152458  1648.825419


In [26]:
df_features = df_features.drop_duplicates(
    subset=['date', 'home_team', 'away_team']
).reset_index(drop=True)

print("Shape after dedup:", df_features.shape)

Shape after dedup: (32306, 11)


In [27]:
df_features_v2 = df_features.merge(
    df_modern_sorted[['date', 'home_team', 'away_team', 'home_elo', 'away_elo']],
    on=['date', 'home_team', 'away_team'],
    how='left'
)

print(df_features_v2.shape)
print(df_features_v2[FEATURES_V2].isnull().sum())

(32306, 13)
home_avg_scored      0
home_avg_conceded    0
away_avg_scored      0
away_avg_conceded    0
is_neutral           0
tournament_weight    0
home_elo             0
away_elo             0
dtype: int64


In [28]:
X2 = df_features_v2[FEATURES_V2]

X2_train, X2_test, yh_train2, yh_test2, ya_train2, ya_test2 = train_test_split(
    X2, df_features_v2['home_score'], df_features_v2['away_score'],
    test_size=0.2, shuffle=False
)

rf_home_v2 = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)
rf_away_v2 = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)
rf_home_v2.fit(X2_train, yh_train2)
rf_away_v2.fit(X2_train, ya_train2)

mae_home_v2 = mean_absolute_error(yh_test2, rf_home_v2.predict(X2_test))
mae_away_v2 = mean_absolute_error(ya_test2, rf_away_v2.predict(X2_test))

print(f"WITH Elo    → Home: {mae_home_v2:.3f}, Away: {mae_away_v2:.3f}")
print(f"WITHOUT Elo → Home: {rf_mae_home:.3f}, Away: {rf_mae_away:.3f}")
print(f"Dummy       → Home: {dummy_mae_home:.3f}, Away: {dummy_mae_away:.3f}")

WITH Elo    → Home: 1.062, Away: 0.846
WITHOUT Elo → Home: 1.135, Away: 0.878
Dummy       → Home: 1.237, Away: 0.933


In [29]:
importance_v2 = pd.DataFrame({
    'feature': FEATURES_V2,
    'importance': rf_home_v2.feature_importances_
}).sort_values('importance', ascending=False)

print(importance_v2)

             feature  importance
3  away_avg_conceded    0.449140
6           home_elo    0.257132
7           away_elo    0.164137
0    home_avg_scored    0.063210
1  home_avg_conceded    0.029887
2    away_avg_scored    0.018441
5  tournament_weight    0.016917
4         is_neutral    0.001136


In [30]:
from xgboost import XGBRegressor

xgb_home = XGBRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    random_state=42
)
xgb_away = XGBRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    random_state=42
)

xgb_home.fit(X2_train, yh_train2)
xgb_away.fit(X2_train, ya_train2)

xgb_mae_home = mean_absolute_error(yh_test2, xgb_home.predict(X2_test))
xgb_mae_away = mean_absolute_error(ya_test2, xgb_away.predict(X2_test))

print(f"XGBoost      → Home: {xgb_mae_home:.3f}, Away: {xgb_mae_away:.3f}")
print(f"RF + Elo     → Home: {mae_home_v2:.3f}, Away: {mae_away_v2:.3f}")
print(f"Dummy        → Home: {dummy_mae_home:.3f}, Away: {dummy_mae_away:.3f}")

XGBoost      → Home: 1.056, Away: 0.840
RF + Elo     → Home: 1.062, Away: 0.846
Dummy        → Home: 1.237, Away: 0.933


In [31]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Poisson needs a specific data shape - one row per TEAM's goal count
poisson_data = pd.concat([
    df_features_v2.assign(
        goals=df_features_v2['home_score'],
        team_elo=df_features_v2['home_elo'],
        opp_elo=df_features_v2['away_elo'],
        is_home=1
    ),
    df_features_v2.assign(
        goals=df_features_v2['away_score'],
        team_elo=df_features_v2['away_elo'],
        opp_elo=df_features_v2['home_elo'],
        is_home=0
    )
])[['goals', 'team_elo', 'opp_elo', 'is_home']]

print(poisson_data.shape)
print(poisson_data.head())

(64612, 4)
   goals  team_elo  opp_elo  is_home
0    5.0    1500.0   1500.0        1
1    3.0    1510.0   1500.0        1
2    2.0    1500.0   1500.0        1
3    2.0    1500.0   1500.0        1
4    2.0    1500.0   1500.0        1


In [32]:
poisson_model = smf.glm(
    formula='goals ~ team_elo + opp_elo + is_home',
    data=poisson_data,
    family=sm.families.Poisson()
).fit()

print(poisson_model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                  goals   No. Observations:                64612
Model:                            GLM   Df Residuals:                    64608
Model Family:                 Poisson   Df Model:                            3
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -97740.
Date:                Tue, 23 Jun 2026   Deviance:                       87515.
Time:                        12:42:41   Pearson chi2:                 9.01e+04
No. Iterations:                     5   Pseudo R-squ. (CS):             0.2571
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.7434      0.045     16.431      0.0

In [33]:
import numpy as np

# Brazil (high elo) at home vs Germany (lower elo)
test_input = pd.DataFrame([{
    'team_elo': 1898.9,   # Brazil
    'opp_elo': 1866.5,    # Germany
    'is_home': 1
}])

expected_goals_brazil = poisson_model.predict(test_input)[0]
print(f"Expected goals (Brazil at home vs Germany): {expected_goals_brazil:.2f}")

# Now flip it - Germany away vs Brazil
test_input2 = pd.DataFrame([{
    'team_elo': 1866.5,   # Germany
    'opp_elo': 1898.9,    # Brazil
    'is_home': 0
}])

expected_goals_germany = poisson_model.predict(test_input2)[0]
print(f"Expected goals (Germany away vs Brazil): {expected_goals_germany:.2f}")

Expected goals (Brazil at home vs Germany): 1.38
Expected goals (Germany away vs Brazil): 0.82


In [34]:
from scipy.stats import poisson

lambda_home = expected_goals_brazil
lambda_away = expected_goals_germany

max_goals = 6  # consider scorelines from 0 to 6 goals per team

# Build probability matrix
score_matrix = np.zeros((max_goals + 1, max_goals + 1))

for h in range(max_goals + 1):
    for a in range(max_goals + 1):
        prob_h = poisson.pmf(h, lambda_home)
        prob_a = poisson.pmf(a, lambda_away)
        score_matrix[h, a] = prob_h * prob_a

score_df = pd.DataFrame(
    score_matrix,
    index=[f"Brazil {i}" for i in range(max_goals + 1)],
    columns=[f"Germany {i}" for i in range(max_goals + 1)]
)

print((score_df * 100).round(2))  # as percentages

          Germany 0  Germany 1  Germany 2  Germany 3  Germany 4  Germany 5  \
Brazil 0      11.07       9.13       3.77       1.04       0.21       0.04   
Brazil 1      15.23      12.56       5.18       1.43       0.29       0.05   
Brazil 2      10.48       8.65       3.57       0.98       0.20       0.03   
Brazil 3       4.81       3.97       1.64       0.45       0.09       0.02   
Brazil 4       1.65       1.36       0.56       0.15       0.03       0.01   
Brazil 5       0.46       0.38       0.15       0.04       0.01       0.00   
Brazil 6       0.10       0.09       0.04       0.01       0.00       0.00   

          Germany 6  
Brazil 0       0.00  
Brazil 1       0.01  
Brazil 2       0.00  
Brazil 3       0.00  
Brazil 4       0.00  
Brazil 5       0.00  
Brazil 6       0.00  


In [35]:
# Flatten and sort to find top 10 most likely scorelines
results = []
for h in range(max_goals + 1):
    for a in range(max_goals + 1):
        results.append((h, a, score_matrix[h, a]))

results_df = pd.DataFrame(results, columns=['brazil_goals', 'germany_goals', 'probability'])
results_df = results_df.sort_values('probability', ascending=False).head(10)
results_df['probability_pct'] = (results_df['probability'] * 100).round(2)

print(results_df[['brazil_goals', 'germany_goals', 'probability_pct']])

    brazil_goals  germany_goals  probability_pct
7              1              0            15.23
8              1              1            12.56
0              0              0            11.07
14             2              0            10.48
1              0              1             9.13
15             2              1             8.65
9              1              2             5.18
21             3              0             4.81
22             3              1             3.97
2              0              2             3.77


In [36]:
p_brazil_win = sum(score_matrix[h, a] for h in range(max_goals+1) for a in range(max_goals+1) if h > a)
p_draw       = sum(score_matrix[h, a] for h in range(max_goals+1) for a in range(max_goals+1) if h == a)
p_germany_win = sum(score_matrix[h, a] for h in range(max_goals+1) for a in range(max_goals+1) if h < a)

print(f"Brazil Win:  {p_brazil_win*100:.1f}%")
print(f"Draw:        {p_draw*100:.1f}%")
print(f"Germany Win: {p_germany_win*100:.1f}%")

Brazil Win:  49.8%
Draw:        27.7%
Germany Win: 22.5%


In [37]:
import pickle

with open('../models/poisson_model.pkl', 'wb') as f:
    pickle.dump(poisson_model, f)

print("Model saved!")

Model saved!


In [38]:
with open('../models/poisson_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

# Sanity check - should match what we got before
test_input = pd.DataFrame([{'team_elo': 1898.9, 'opp_elo': 1866.5, 'is_home': 1}])
print(loaded_model.predict(test_input)[0])

1.376301153545154


In [39]:
import json

with open('../models/team_elo_ratings.json', 'w') as f:
    json.dump(final_elo, f, indent=2)

print(f"Saved ratings for {len(final_elo)} teams")

Saved ratings for 326 teams


In [42]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import json
import pickle

# Load base features (no elo columns)
df_features_v2 = pd.read_csv('../data/features.csv')
df_features_v2['date'] = pd.to_datetime(df_features_v2['date'])

print("features.csv columns:", list(df_features_v2.columns))
print("Shape:", df_features_v2.shape)

features.csv columns: ['date', 'home_team', 'away_team', 'home_avg_scored', 'home_avg_conceded', 'away_avg_scored', 'away_avg_conceded', 'is_neutral', 'tournament_weight', 'home_score', 'away_score']
Shape: (32307, 11)


In [43]:
df_modern_sorted = df_features_v2.sort_values('date').reset_index(drop=True)

def calculate_elo_ratings(df, k=20, initial_rating=1500):
    elo = {}
    home_elo_list = []
    away_elo_list = []
    for _, row in df.iterrows():
        home = row['home_team']
        away = row['away_team']
        r_home = elo.get(home, initial_rating)
        r_away = elo.get(away, initial_rating)
        home_elo_list.append(r_home)
        away_elo_list.append(r_away)
        e_home = 1 / (1 + 10 ** ((r_away - r_home) / 400))
        e_away = 1 - e_home
        if row['home_score'] > row['away_score']:
            actual_home, actual_away = 1.0, 0.0
        elif row['home_score'] < row['away_score']:
            actual_home, actual_away = 0.0, 1.0
        else:
            actual_home, actual_away = 0.5, 0.5
        elo[home] = r_home + k * (actual_home - e_home)
        elo[away] = r_away + k * (actual_away - e_away)
    df['home_elo'] = home_elo_list
    df['away_elo'] = away_elo_list
    return df, elo

df_modern_sorted, final_elo = calculate_elo_ratings(df_modern_sorted)
print("Columns:", list(df_modern_sorted.columns))
print("Shape:", df_modern_sorted.shape)

Columns: ['date', 'home_team', 'away_team', 'home_avg_scored', 'home_avg_conceded', 'away_avg_scored', 'away_avg_conceded', 'is_neutral', 'tournament_weight', 'home_score', 'away_score', 'home_elo', 'away_elo']
Shape: (32307, 13)


In [44]:
home_rows = df_modern_sorted.assign(
    goals            = df_modern_sorted['home_score'],
    team_elo         = df_modern_sorted['home_elo'],
    opp_elo          = df_modern_sorted['away_elo'],
    avg_scored       = df_modern_sorted['home_avg_scored'],
    avg_conceded     = df_modern_sorted['home_avg_conceded'],
    opp_avg_scored   = df_modern_sorted['away_avg_scored'],
    opp_avg_conceded = df_modern_sorted['away_avg_conceded'],
    is_home          = 1,
)

away_rows = df_modern_sorted.assign(
    goals            = df_modern_sorted['away_score'],
    team_elo         = df_modern_sorted['away_elo'],
    opp_elo          = df_modern_sorted['home_elo'],
    avg_scored       = df_modern_sorted['away_avg_scored'],
    avg_conceded     = df_modern_sorted['away_avg_conceded'],
    opp_avg_scored   = df_modern_sorted['home_avg_scored'],
    opp_avg_conceded = df_modern_sorted['home_avg_conceded'],
    is_home          = 0,
)

POISSON_FEATURES = [
    'goals', 'team_elo', 'opp_elo', 'is_home',
    'avg_scored', 'avg_conceded',
    'opp_avg_scored', 'opp_avg_conceded',
    'tournament_weight'
]

poisson_data_v2 = pd.concat([
    home_rows[POISSON_FEATURES],
    away_rows[POISSON_FEATURES]
]).dropna().reset_index(drop=True)

print("Shape:", poisson_data_v2.shape)
print(poisson_data_v2.head(3))

Shape: (64614, 9)
   goals  team_elo  opp_elo  is_home  avg_scored  avg_conceded  \
0    5.0    1500.0   1500.0        1         1.0           1.0   
1    3.0    1510.0   1500.0        1         5.0           0.0   
2    2.0    1500.0   1500.0        1         1.0           1.0   

   opp_avg_scored  opp_avg_conceded  tournament_weight  
0             1.0               1.0                0.5  
1             1.0               1.0                0.5  
2             1.0               1.0                0.5  


In [45]:
import statsmodels.formula.api as smf
import statsmodels.api as sm

poisson_model_v2 = smf.glm(
    formula='''goals ~ team_elo + opp_elo + is_home
                     + avg_scored + avg_conceded
                     + opp_avg_scored + opp_avg_conceded
                     + tournament_weight''',
    data=poisson_data_v2,
    family=sm.families.Poisson()
).fit()

print(poisson_model_v2.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                  goals   No. Observations:                64614
Model:                            GLM   Df Residuals:                    64605
Model Family:                 Poisson   Df Model:                            8
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -95266.
Date:                Tue, 23 Jun 2026   Deviance:                       82561.
Time:                        12:46:36   Pearson chi2:                 7.92e+04
No. Iterations:                     6   Pseudo R-squ. (CS):             0.3120
Covariance Type:            nonrobust                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            -0.1594      0.05

In [46]:
import pickle

with open('../models/poisson_model.pkl', 'wb') as f:
    pickle.dump(poisson_model_v2, f)

print("Saved!")

with open('../models/poisson_model.pkl', 'rb') as f:
    test_load = pickle.load(f)

test_input = pd.DataFrame([{
    'team_elo': 1975.2, 'opp_elo': 1898.9, 'is_home': 1,
    'avg_scored': 2.1, 'avg_conceded': 0.8,
    'opp_avg_scored': 2.5, 'opp_avg_conceded': 0.6,
    'tournament_weight': 3.0
}])
print("Test prediction:", test_load.predict(test_input)[0])

Saved!
Test prediction: 1.6576772917077391
